In [ ]:
# --- DEVELOPED BY: Yoginder Syal ---
# Goal: Establish the technical architecture and data types for MOP compatibility.

import pandas as pd
import numpy as np

def load_and_initialize(file_path):
    """Initial ingestion and metadata standardization."""
    df = pd.read_csv(file_path)
    
    # Standardizing column names for PostgreSQL/Supabase compatibility [cite: 317, 349]
    df.columns = [c.lower().replace(' ', '_') for c in df.columns]
    
    # Converting timestamps to ISO 8601 format
    df['arrival_time'] = pd.to_datetime(df['arrival_time'])
    df['departure_time'] = pd.to_datetime(df['departure_time'])
    
    return df

def normalize_sensor_states(df):
    """Mapping categorical occupancy to binary for future ML modeling."""
    # Present = 1, Vacant/Unoccupied = 0
    status_map = {'Present': 1, 'Vacant': 0, 'Unoccupied': 0}
    df['occupancy_binary'] = df['status'].map(status_map).fillna(0).astype(int)
    return df

# Initialize base dataframe
raw_data_path = 'melbourne_parking_2020.csv' # Dataset 2
df_base = load_and_initialize(raw_data_path)
df_normalized = normalize_sensor_states(df_base)

print(f"Initial setup complete. Row count: {len(df_normalized)}")

In [ ]:
# --- DEVELOPED BY: Atishay Jain ---
# Goal: Align technical output with stakeholder goals and handle real-world outliers.

def apply_strategic_filters(df):
    """Applying business logic to focus on high-impact congestion zones."""
    
    # 1. Handling COVID-19 Anomalies (March - May 2020)
    # Strategic decision to exclude lockdown data to prevent model skewing.
    lockdown_mask = (df['arrival_time'] >= '2020-03-23') & (df['arrival_time'] <= '2020-05-31')
    df_clean = df[~lockdown_mask].copy()
    
    # 2. Targeting High-Congestion CBD Zones
    # Focus on streets identified in our project scope for the City of Melbourne.
    target_streets = ['Lonsdale St', 'Bourke St', 'Collins St', 'Elizabeth St']
    df_final = df_clean[df_clean['street_name'].isin(target_streets)].copy()
    
    return df_final

def handle_missing_values(df):
    """Imputation strategy to maintain time-series integrity."""
    # Forward-fill short gaps in sensor data to maintain continuity
    df = df.sort_values(by=['bay_id', 'arrival_time'])
    df['occupancy_binary'] = df.groupby('bay_id')['occupancy_binary'].ffill()
    return df

# Execute Lead Logic
df_strategic = apply_strategic_filters(df_normalized)
df_final = handle_missing_values(df_strategic)

print(f"Strategic filtering complete. Rows remaining: {len(df_final)}")
df_final.head()